In [1]:
# Import polars for parquet processing and set Debug level do verbose
import polars as pl
pl.Config.set_verbose(True)

polars.config.Config

In [6]:
# Scan the parquet file, set the following schema to greater reduce memory usage
active_repos_cumulative_stats_by_day = pl.scan_parquet(source='daily_cumulative_stats_no_inactive_repo.parquet',
                                                       schema={
                                             "repo_name": pl.String,
                                             "day": pl.Date,
                                             "total_stars": pl.UInt32,
                                             "total_forks": pl.UInt32,
                                             "total_issues_opened": pl.UInt32,
                                             "total_issues_closed": pl.UInt32,
                                             "total_prs_opened": pl.UInt32,
                                             "total_prs_merged": pl.UInt32,
                                             "total_commits": pl.UInt32,
                                             "total_comments": pl.UInt32
                                         })

_init_credential_provider_builder(): credential_provider_init = None


In [7]:
distinct_active_repository_names = active_repos_cumulative_stats_by_day.select('repo_name').unique().collect()

In [8]:
sampled_repository_names = distinct_active_repository_names.sample(n=2_000_000, with_replacement=False)

In [18]:
# Free up memory
del distinct_active_repository_names

NameError: name 'unique_repos' is not defined

In [10]:
# Keep only sampled repositories (by inner joining the samples with the original dataframe)
active_sampled_repositories = active_repos_cumulative_stats_by_day.collect().join(other=sampled_repository_names, on='repo_name', how='inner',
                                                                                  maintain_order='left')

In [19]:
# Free up memory
del sampled_repository_names

In [ ]:
# Checkpoint save
active_sampled_repositories.write_csv('sampled_daily_2M_repos_sorted.csv')

In [6]:
# Checkpoint load
active_sampled_repositories = pl.read_csv('sampled_daily_3M_repos_sorted.csv')

In [7]:
metric_cols = [
    'total_stars', 'total_forks', 'total_issues_opened',
    'total_issues_closed', 'total_prs_opened', 'total_prs_merged',
    'total_commits', 'total_comments'
]

In [8]:
# Mention set_sorted by repo_name and day, as original data has this property. Boosts memory and cpu optimisation.
active_sampled_repositories.set_sorted(['repo_name', 'day'])

repo_name,day,total_stars,total_forks,total_issues_opened,total_issues_closed,total_prs_opened,total_prs_merged,total_commits,total_comments
str,str,i64,i64,i64,i64,i64,i64,i64,i64
"""/AHEasing""","""2013-05-10""",1,0,0,0,0,0,0,0
"""/ANE-Push-Notification""","""2013-04-20""",1,0,0,0,0,0,0,0
"""/AVFoundationDemos""","""2013-03-30""",1,0,0,0,0,0,0,0
"""/AndroidFFmpeg""","""2013-03-21""",0,0,1,0,0,0,0,0
"""/AndroidFFmpeg""","""2013-05-04""",1,0,1,0,0,0,0,0
…,…,…,…,…,…,…,…,…,…
"""zzzzzzzzzzz0/zsp-go""","""2019-12-25""",1,1,0,0,0,0,50,0
"""zzzzzzzzzzzzzoe/ZoeLaunchScree…","""2016-12-12""",1,0,0,0,0,0,9,0
"""zzzzzzzzzzzzzoe/ZoeLaunchScree…","""2017-12-20""",2,0,0,0,0,0,9,0


In [9]:
repositories_with_rolling_metrics = (active_sampled_repositories
                                     # Cast to Float (requirement for all subsequent math)
                                     .with_columns([pl.col(c).cast(pl.Float32).alias(c) for c in metric_cols])

                                     # Lags and Rolling Statistics
                                     .with_columns(
    # Lags (1, 7  days)
    [pl.col(c).shift(1).over('repo_name').alias(f"{c}_lag_1d") for c in metric_cols] +
    [pl.col(c).shift(7).over('repo_name').alias(f"{c}_lag_7d") for c in metric_cols] +

    # Growth Rates (Pct Change; 1, 7  days)
    [pl.col(c).pct_change(n=1).over('repo_name').alias(f"{c}_growth_1d") for c in metric_cols] +
    [pl.col(c).pct_change(n=7).over('repo_name').alias(f"{c}_growth_7d") for c in metric_cols] +

    # Rolling Means (7, 30 days)
    [pl.col(c).rolling_mean(window_size=7).over('repo_name').alias(f"{c}_rolling_mean_7d") for c in metric_cols] +
    [pl.col(c).rolling_mean(window_size=30).over('repo_name').alias(f"{c}_rolling_mean_30d") for c in metric_cols] +

    # Rolling Median (7, 30 days)
    [pl.col(c).rolling_median(window_size=7).over('repo_name').alias(f"{c}_rolling_median_7d") for c in metric_cols] +
    [pl.col(c).rolling_median(window_size=30).over('repo_name').alias(f"{c}_rolling_median_30d") for c in metric_cols] +

    # Rolling Stds (7, 30 days)
    [pl.col(c).rolling_std(window_size=7).over('repo_name').alias(f"{c}_rolling_std_7d") for c in metric_cols] +
    [pl.col(c).rolling_std(window_size=30).over('repo_name').alias(f"{c}_rolling_std_30d") for c in metric_cols] +

    # Slope (14 days)
    [((pl.col(c) - pl.col(c).shift(14)) / 14).over('repo_name').alias(f"{c}_slope_14d") for c in metric_cols]
)

                                     # 4. Net Change (Row-wise math, NO .over needed here as columns are aligned)
                                     .with_columns([
    (pl.col(c) - pl.col(f"{c}_lag_1d")).alias(f"{c}_daily_change")
    for c in metric_cols
])

                                     # 5. Cleanup
                                     .with_columns([
    pl.when(pl.col(pl.Float32).is_infinite())
    .then(None)
    .otherwise(pl.col(pl.Float32))
    .name.keep()
])
                                     .fill_nan(0).fill_null(0)
                                     )

In [10]:
# Logarithmic transformation to stabilize variance and reduce data skewness.
repositories_with_log1p_metrics = repositories_with_rolling_metrics.with_columns([
    pl.col(c).log1p().alias(f"{c}_log1p")
    for c in repositories_with_rolling_metrics.columns if
    ("growth" not in c) and ("slope" not in c) and ("day" not in c) and ("repo_name" not in c)
])

In [11]:
# Free up memory
del repositories_with_rolling_metrics

In [12]:
cols_to_scale = [c for c in repositories_with_log1p_metrics.columns if c.endswith('_log1p')]

repositories_with_scaled_rolling_metrics = repositories_with_log1p_metrics.with_columns([
    (
        #maxabsscaler
            pl.col(c) / (pl.col(c).max() + 1e-6)
    ).alias(c.replace('_log1p', '_scaled'))
    for c in cols_to_scale
])

In [13]:
del repositories_with_log1p_metrics

In [14]:
repositories_with_relevant_metrics = repositories_with_scaled_rolling_metrics.select(
    pl.col("repo_name"),
    pl.col("day"),
    pl.col(metric_cols),
    pl.col("^.*_scaled$")
)

In [16]:
print("Starting processing... this may take a while.")

repositories_with_relevant_metrics.write_csv(
    "processed_github_features.csv",
    # maintain_order=False
)
print("Done! File saved.")

Starting processing... this may take a while.


_init_credential_provider_builder(): credential_provider_init = None


Done! File saved.


In [ ]:
print("Starting processing... this may take a while.")

repositories_with_relevant_metrics.write_parquet('processed_github_features.parquet')

print("Done! File saved.")